# **Google Drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
from pathlib import Path

# Check dataset directories
dataset_paths = [
    DATASET_DIR / "Train" / "Image",
    DATASET_DIR / "Train" / "GT_Object",
    DATASET_DIR / "Test" / "Image",
    DATASET_DIR / "Test" / "GT_Object",
]

print("DATASET CHECK")
for path in dataset_paths:
    print(f"{path.name}: {'EXISTS' if path.exists() else 'MISSING'}")

# Check existing project files
project_paths = [
    PROJECT_DIR / "dataset.py",
    PROJECT_DIR / "requirements.txt",
    PROJECT_DIR / "README.md",
    PROJECT_DIR / "models",
    PROJECT_DIR / "scripts",
    PROJECT_DIR / "notebooks",
]

print("\nPROJECT CHECK")
for path in project_paths:
    print(f"{path.name}: {'EXISTS' if path.exists() else 'MISSING'}")

DATASET CHECK
Image: EXISTS
GT_Object: EXISTS
Image: EXISTS
GT_Object: EXISTS

PROJECT CHECK
dataset.py: EXISTS
requirements.txt: EXISTS
README.md: EXISTS
models: EXISTS
scripts: EXISTS
notebooks: EXISTS


In [9]:
import torch
from torch.utils.data import DataLoader
import sys

sys.path.insert(0, str(PROJECT_DIR))

from dataset import COD10KDataset

train_dataset = COD10KDataset(
    root_dir=DATASET_DIR,
    split="Train",
    image_size=(352, 352)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

images, masks = next(iter(train_loader))

print("IMAGE CHECK")
print("Shape:", images.shape)
print("Data type:", images.dtype)
print("Minimum:", images.min().item())
print("Maximum:", images.max().item())

print("\nMASK CHECK")
print("Shape:", masks.shape)
print("Data type:", masks.dtype)
print("Minimum:", masks.min().item())
print("Maximum:", masks.max().item())
print("Unique values in first mask:",
      torch.unique(masks[0]).tolist())

IMAGE CHECK
Shape: torch.Size([4, 3, 352, 352])
Data type: torch.float32
Minimum: 0.0
Maximum: 1.0

MASK CHECK
Shape: torch.Size([4, 1, 352, 352])
Data type: torch.float32
Minimum: 0.0
Maximum: 1.0
Unique values in first mask: [0.0, 1.0]


# **Test pretrained ResNet34**

In [10]:
import torch
import torch.nn as nn
from torchvision.models import resnet34, ResNet34_Weights

# Select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load ResNet34 with pretrained ImageNet weights
weights = ResNet34_Weights.DEFAULT
encoder = resnet34(weights=weights)

# Remove the classification head
encoder = nn.Sequential(*list(encoder.children())[:-2])

encoder = encoder.to(device)
encoder.eval()

# Create a sample input
dummy_input = torch.randn(1, 3, 352, 352).to(device)

# Extract features
with torch.no_grad():

    x = dummy_input

    features = []

    for index, layer in enumerate(encoder):

        x = layer(x)

        if index in [3, 4, 5, 6, 7]:
            features.append(x)

# Display feature dimensions
print("RESNET34 ENCODER CHECK")

for i, feature in enumerate(features):
    print(f"Feature {i + 1}: {feature.shape}")

print("\nEncoder device:", next(encoder.parameters()).device)
print("Pretrained weights loaded successfully.")

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 136MB/s]


RESNET34 ENCODER CHECK
Feature 1: torch.Size([1, 64, 88, 88])
Feature 2: torch.Size([1, 64, 88, 88])
Feature 3: torch.Size([1, 128, 44, 44])
Feature 4: torch.Size([1, 256, 22, 22])
Feature 5: torch.Size([1, 512, 11, 11])

Encoder device: cuda:0
Pretrained weights loaded successfully.


# **Unet.py**

In [12]:
from pathlib import Path

model_file = PROJECT_DIR / "models" / "unet.py"

model_code = r'''
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision.models import resnet34, ResNet34_Weights


class ConvBlock(nn.Module):
    """Two convolution layers for feature refinement."""

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    """Upsample features, combine skip connection and refine."""

    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.conv = ConvBlock(
            in_channels + skip_channels,
            out_channels
        )

    def forward(self, x, skip=None):

        if skip is not None:
            x = F.interpolate(
                x,
                size=skip.shape[2:],
                mode="bilinear",
                align_corners=False
            )

            x = torch.cat([x, skip], dim=1)

        return self.conv(x)


class ResNet34UNet(nn.Module):
    """U-Net with an ImageNet-pretrained ResNet34 encoder."""

    def __init__(self, pretrained=True):
        super().__init__()

        weights = ResNet34_Weights.DEFAULT if pretrained else None

        backbone = resnet34(weights=weights)

        # ResNet34 encoder
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        # U-Net decoder
        self.decoder1 = DecoderBlock(512, 256, 256)
        self.decoder2 = DecoderBlock(256, 128, 128)
        self.decoder3 = DecoderBlock(128, 64, 64)
        self.decoder4 = DecoderBlock(64, 64, 64)

        # Final full-resolution refinement
        self.final_conv = ConvBlock(64, 32)

        # One output channel for binary segmentation
        self.output_conv = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):

        input_size = x.shape[2:]

        # Encoder
        stem = self.relu(self.bn1(self.conv1(x)))
        # 64 x 176 x 176

        x = self.maxpool(stem)

        enc1 = self.layer1(x)
        # 64 x 88 x 88

        enc2 = self.layer2(enc1)
        # 128 x 44 x 44

        enc3 = self.layer3(enc2)
        # 256 x 22 x 22

        enc4 = self.layer4(enc3)
        # 512 x 11 x 11

        # Decoder with skip connections
        x = self.decoder1(enc4, enc3)
        x = self.decoder2(x, enc2)
        x = self.decoder3(x, enc1)
        x = self.decoder4(x, stem)

        # Upsample to original input dimensions
        x = F.interpolate(
            x,
            size=input_size,
            mode="bilinear",
            align_corners=False
        )

        x = self.final_conv(x)

        # Return raw logits (no sigmoid here)
        return self.output_conv(x)
'''

model_file.write_text(model_code.strip() + "\n")

print("Model file created successfully:")
print(model_file)

Model file created successfully:
/content/drive/MyDrive/Camouflaged-Wildlife-Detection-System/models/unet.py


In [13]:
import sys
import torch

sys.path.insert(0, str(PROJECT_DIR))

from models.unet import ResNet34UNet

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model
model = ResNet34UNet(pretrained=True).to(device)

# Create dummy input
dummy_input = torch.randn(2, 3, 352, 352).to(device)

# Forward pass
model.train()

output = model(dummy_input)

print("MODEL TEST")
print("Input shape:", dummy_input.shape)
print("Output shape:", output.shape)
print("Output data type:", output.dtype)
print("Model device:", next(model.parameters()).device)

# Check output dimensions
assert output.shape == (2, 1, 352, 352), "Incorrect output dimensions!"

print("\nOutput dimensions: PASSED")

# Gradient check
dummy_target = torch.rand(2, 1, 352, 352).to(device)

loss = torch.nn.functional.binary_cross_entropy_with_logits(
    output,
    dummy_target
)

loss.backward()

print("Gradient check:", "PASSED" if model.output_conv.weight.grad is not None else "FAILED")
print("Test loss:", loss.item())

MODEL TEST
Input shape: torch.Size([2, 3, 352, 352])
Output shape: torch.Size([2, 1, 352, 352])
Output data type: torch.float32
Model device: cuda:0

Output dimensions: PASSED
Gradient check: PASSED
Test loss: 0.7087166905403137


# **losses.py**

In [14]:
from pathlib import Path

utils_dir = PROJECT_DIR / "utils"
utils_dir.mkdir(exist_ok=True)

loss_file = utils_dir / "losses.py"

loss_code = r'''
import torch
import torch.nn as nn


class DiceLoss(nn.Module):
    """Dice loss for binary segmentation."""

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):

        probabilities = torch.sigmoid(logits)

        probabilities = probabilities.contiguous().view(-1)
        targets = targets.contiguous().view(-1)

        intersection = (probabilities * targets).sum()

        dice_score = (
            (2.0 * intersection + self.smooth)
            /
            (probabilities.sum() + targets.sum() + self.smooth)
        )

        return 1.0 - dice_score


class CombinedLoss(nn.Module):
    """Equal-weight combination of BCE and Dice loss."""

    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):

        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)

        total_loss = (
            self.bce_weight * bce_loss
            +
            self.dice_weight * dice_loss
        )

        return total_loss
'''

loss_file.write_text(loss_code.strip() + "\n")

print("Loss functions created successfully:")
print(loss_file)

Loss functions created successfully:
/content/drive/MyDrive/Camouflaged-Wildlife-Detection-System/utils/losses.py


In [15]:
import sys
import torch

sys.path.insert(0, str(PROJECT_DIR))

from utils.losses import DiceLoss, CombinedLoss

# Use our existing model
model.eval()

# Generate sample input and target
dummy_input = torch.randn(2, 3, 352, 352).to(device)
dummy_target = torch.randint(
    0, 2, (2, 1, 352, 352)
).float().to(device)

# Generate model predictions
model_output = model(dummy_input)

# Initialize losses
dice_loss_fn = DiceLoss()
combined_loss_fn = CombinedLoss()

# Calculate losses
dice_value = dice_loss_fn(model_output, dummy_target)
combined_value = combined_loss_fn(model_output, dummy_target)

print("LOSS FUNCTION TEST")
print("Dice Loss:", dice_value.item())
print("Combined Loss:", combined_value.item())

# Gradient test
model.zero_grad()
combined_value.backward()

print(
    "Gradient check:",
    "PASSED" if model.output_conv.weight.grad is not None else "FAILED"
)

# Basic validation
assert torch.isfinite(dice_value), "Dice loss is not finite"
assert torch.isfinite(combined_value), "Combined loss is not finite"
assert model.output_conv.weight.grad is not None

print("\nAll loss tests passed!")

LOSS FUNCTION TEST
Dice Loss: 0.48361736536026
Combined Loss: 0.5896254181861877
Gradient check: PASSED

All loss tests passed!


# **Training and Validation Split**

In [17]:
import torch
from torch.utils.data import random_split, DataLoader, Dataset
from torchvision import transforms

# Reproducibility
SEED = 42
torch.manual_seed(SEED)

# Load original training dataset
full_train_dataset = COD10KDataset(
    root_dir=DATASET_DIR,
    split="Train",
    image_size=(352, 352)
)

# Calculate split sizes
train_size = 5400
val_size = 600

# Create reproducible split
train_subset, val_subset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# ImageNet normalization
imagenet_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# Wrapper to normalize images while leaving masks unchanged
class NormalizedDataset(Dataset):

    def __init__(self, subset):
        self.subset = subset

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, index):
        image, mask = self.subset[index]

        image = imagenet_normalize(image)

        return image, mask


train_dataset = NormalizedDataset(train_subset)
val_dataset = NormalizedDataset(val_subset)

# Create DataLoaders
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Display results
print("DATASET SPLIT")
print("Original training images:", len(full_train_dataset))
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

print("\nDATALOADER CHECK")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Batch size:", BATCH_SIZE)

DATASET SPLIT
Original training images: 6000
Training images: 5400
Validation images: 600

DATALOADER CHECK
Training batches: 675
Validation batches: 75
Batch size: 8


In [18]:
images, masks = next(iter(train_loader))

print("NORMALIZATION CHECK")
print("Image shape:", images.shape)
print("Image mean:", images.mean().item())
print("Image standard deviation:", images.std().item())

print("\nMASK CHECK")
print("Mask shape:", masks.shape)
print("Mask minimum:", masks.min().item())
print("Mask maximum:", masks.max().item())
print("Unique mask values:", torch.unique(masks).tolist())

NORMALIZATION CHECK
Image shape: torch.Size([8, 3, 352, 352])
Image mean: -0.40332889556884766
Image standard deviation: 1.1292071342468262

MASK CHECK
Mask shape: torch.Size([8, 1, 352, 352])
Mask minimum: 0.0
Mask maximum: 1.0
Unique mask values: [0.0, 1.0]


# **Optimizer**

In [19]:
import torch

# Initialize a fresh model for training
model = ResNet34UNet(pretrained=True).to(device)

# Separate encoder and decoder parameters
encoder_params = (
    list(model.conv1.parameters())
    + list(model.bn1.parameters())
    + list(model.layer1.parameters())
    + list(model.layer2.parameters())
    + list(model.layer3.parameters())
    + list(model.layer4.parameters())
)

decoder_params = (
    list(model.decoder1.parameters())
    + list(model.decoder2.parameters())
    + list(model.decoder3.parameters())
    + list(model.decoder4.parameters())
    + list(model.final_conv.parameters())
    + list(model.output_conv.parameters())
)

# AdamW optimizer
optimizer = torch.optim.AdamW(
    [
        {
            "params": encoder_params,
            "lr": 1e-4
        },
        {
            "params": decoder_params,
            "lr": 1e-3
        }
    ],
    weight_decay=1e-4
)

# Combined loss
criterion = CombinedLoss(
    bce_weight=0.5,
    dice_weight=0.5
)

print("TRAINING CONFIGURATION")
print("Optimizer:", optimizer.__class__.__name__)
print("Encoder learning rate:", optimizer.param_groups[0]["lr"])
print("Decoder learning rate:", optimizer.param_groups[1]["lr"])
print("Weight decay:", optimizer.param_groups[0]["weight_decay"])
print("Loss function:", criterion.__class__.__name__)
print("Total trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

TRAINING CONFIGURATION
Optimizer: AdamW
Encoder learning rate: 0.0001
Decoder learning rate: 0.001
Weight decay: 0.0001
Loss function: CombinedLoss
Total trainable parameters: 24521697


# **Training Smoke Test**

In [20]:
import torch

model.train()
optimizer.zero_grad(set_to_none=True)

# Get one real training batch
images, masks = next(iter(train_loader))
images = images.to(device, non_blocking=True)
masks = masks.to(device, non_blocking=True)

# Reset GPU memory tracking
torch.cuda.reset_peak_memory_stats()

# Forward pass and loss calculation
with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
    logits = model(images)
    loss = criterion(logits, masks)

# Backpropagation test
scaler = torch.amp.GradScaler("cuda")
scaler.scale(loss).backward()

# Clear gradients without updating model weights
optimizer.zero_grad(set_to_none=True)

torch.cuda.synchronize()

print("TRAINING SMOKE TEST")
print("Input shape:", images.shape)
print("Output shape:", logits.shape)
print("Loss:", round(loss.item(), 6))
print("Loss is finite:", torch.isfinite(loss).item())
print("Backward pass: PASSED")
print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)
print("Model weights updated: No")

TRAINING SMOKE TEST
Input shape: torch.Size([8, 3, 352, 352])
Output shape: torch.Size([8, 1, 352, 352])
Loss: 0.776516
Loss is finite: True
Backward pass: PASSED
Peak GPU memory: 1.67 GB
Model weights updated: No


# **Training**

In [ ]:

import torch
import csv
from pathlib import Path
from tqdm.auto import tqdm

# Training configuration
NUM_EPOCHS = 20
USE_AMP = torch.cuda.is_available()

# Save locations
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Camouflaged-Wildlife-Detection-System"
)
CHECKPOINT_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_model.pth"
LATEST_MODEL_PATH = CHECKPOINT_DIR / "latest_model.pth"
HISTORY_PATH = RESULTS_DIR / "training_history.csv"

# Mixed precision scaler
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

best_val_loss = float("inf")
history = []

print("STARTING MODEL TRAINING")
print("Epochs:", NUM_EPOCHS)
print("Mixed precision:", USE_AMP)
print("Device:", device)
print("Best checkpoint:", BEST_MODEL_PATH)
print("Latest checkpoint:", LATEST_MODEL_PATH)

for epoch in range(NUM_EPOCHS):

    # -------------------------
    # TRAINING
    # -------------------------
    model.train()
    running_train_loss = 0.0

    train_progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} - Training"
    )

    for images, masks in train_progress:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            outputs = model(images)

        loss = criterion(outputs.float(), masks.float())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_train_loss += loss.item()
        train_progress.set_postfix(loss=f"{loss.item():.4f}")

    epoch_train_loss = running_train_loss / len(train_loader)

    # -------------------------
    # VALIDATION
    # -------------------------
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():

        val_progress = tqdm(
            val_loader,
            desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} - Validation"
        )

        for images, masks in val_progress:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):
                outputs = model(images)

            loss = criterion(outputs.float(), masks.float())
            running_val_loss += loss.item()

            val_progress.set_postfix(loss=f"{loss.item():.4f}")

    epoch_val_loss = running_val_loss / len(val_loader)

    # -------------------------
    # RECORD EPOCH RESULTS
    # -------------------------
    epoch_record = {
        "epoch": epoch + 1,
        "train_loss": epoch_train_loss,
        "val_loss": epoch_val_loss
    }

    history.append(epoch_record)

    # -------------------------
    # SAVE LATEST CHECKPOINT
    # Runs after EVERY completed epoch
    # -------------------------
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": epoch_train_loss,
        "val_loss": epoch_val_loss,
        "architecture": "ResNet34UNet",
        "image_size": 352
    }, LATEST_MODEL_PATH)

    # -------------------------
    # UPDATE BEST CHECKPOINT
    # Only when validation loss improves
    # -------------------------
    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "architecture": "ResNet34UNet",
            "image_size": 352
        }, BEST_MODEL_PATH)

        checkpoint_status = "NEW BEST MODEL SAVED"

    else:
        checkpoint_status = "PREVIOUS BEST MODEL RETAINED"

    # -------------------------
    # SAVE TRAINING HISTORY
    # -------------------------
    with open(HISTORY_PATH, "w", newline="") as file:

        writer = csv.DictWriter(
            file,
            fieldnames=["epoch", "train_loss", "val_loss"]
        )

        writer.writeheader()
        writer.writerows(history)

    # -------------------------
    # EPOCH SUMMARY
    # -------------------------
    print(f"\nEPOCH {epoch + 1}/{NUM_EPOCHS}")
    print(f"Training loss:   {epoch_train_loss:.6f}")
    print(f"Validation loss: {epoch_val_loss:.6f}")
    print(f"Best val loss:   {best_val_loss:.6f}")
    print("Latest checkpoint: SAVED")
    print("Best checkpoint:", checkpoint_status)
    print("-" * 45)

# -------------------------
# FINAL TRAINING SUMMARY
# -------------------------
print("\nTRAINING FINISHED")

print("Best validation loss:", best_val_loss)
print("Best checkpoint exists:", BEST_MODEL_PATH.exists())
print("Latest checkpoint exists:", LATEST_MODEL_PATH.exists())
print("Training history exists:", HISTORY_PATH.exists())

STARTING MODEL TRAINING
Epochs: 20
Mixed precision: True
Device: cuda
Best checkpoint: /content/drive/MyDrive/Camouflaged-Wildlife-Detection-System/models/best_model.pth
Latest checkpoint: /content/drive/MyDrive/Camouflaged-Wildlife-Detection-System/models/latest_model.pth


Epoch 1/20 - Training:   0%|          | 0/675 [00:00<?, ?it/s]

# **Resume Training (If needed)**

In [ ]:

import torch
import csv
import math
from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, Sampler

# ==================================================
# 1. TRAINING CONFIGURATION
# ==================================================

NUM_EPOCHS = 20
USE_AMP = torch.cuda.is_available()

SAVE_EVERY_BATCHES = 50
SEED = 42

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Camouflaged-Wildlife-Detection-System"
)

CHECKPOINT_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_model.pth"
LATEST_MODEL_PATH = CHECKPOINT_DIR / "latest_model.pth"
RESUME_PATH = CHECKPOINT_DIR / "resume_checkpoint.pth"
HISTORY_PATH = RESULTS_DIR / "training_history.csv"

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


# ==================================================
# 2. DETERMINISTIC RESUMABLE SAMPLER
# ==================================================

class ResumableRandomSampler(Sampler):

    def __init__(self, data_source, seed, epoch, start_index=0):
        self.data_source = data_source
        self.seed = seed
        self.epoch = epoch
        self.start_index = start_index

    def __iter__(self):
        generator = torch.Generator()
        generator.manual_seed(self.seed + self.epoch)

        indices = torch.randperm(
            len(self.data_source),
            generator=generator
        ).tolist()

        return iter(indices[self.start_index:])

    def __len__(self):
        return len(self.data_source) - self.start_index


# ==================================================
# 3. SAFE CHECKPOINT SAVING
# ==================================================

def save_resume_checkpoint(
    epoch,
    next_batch,
    running_train_loss,
    batches_done,
    best_val_loss,
    history
):

    checkpoint = {
        "epoch": epoch,
        "next_batch": next_batch,
        "running_train_loss": running_train_loss,
        "batches_done": batches_done,
        "best_val_loss": best_val_loss,
        "history": history,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "architecture": "ResNet34UNet",
        "image_size": 352,
        "seed": SEED
    }

    temporary_path = CHECKPOINT_DIR / "resume_checkpoint.tmp"

    torch.save(checkpoint, temporary_path)
    temporary_path.replace(RESUME_PATH)


# ==================================================
# 4. RESTORE PREVIOUS TRAINING STATE
# ==================================================

start_epoch = 1
start_batch = 0
running_train_loss = 0.0
batches_done = 0
best_val_loss = float("inf")
history = []

if RESUME_PATH.exists():

    print("RECOVERY CHECKPOINT FOUND")
    print("Restoring previous training session...")

    checkpoint = torch.load(
        RESUME_PATH,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])

    start_epoch = checkpoint["epoch"]
    start_batch = checkpoint["next_batch"]
    running_train_loss = checkpoint["running_train_loss"]
    batches_done = checkpoint["batches_done"]
    best_val_loss = checkpoint["best_val_loss"]
    history = checkpoint["history"]

    print("Recovery successful!")
    print("Resuming epoch:", start_epoch)
    print("Next batch:", start_batch + 1)

else:
    print("No previous checkpoint found.")
    print("Starting fresh training.")


# ==================================================
# 5. ACTUAL TRAINING
# ==================================================

print("\nTRAINING CONFIGURATION")
print("Total epochs:", NUM_EPOCHS)
print("Starting epoch:", start_epoch)
print("Mixed precision:", USE_AMP)
print("Save interval:", SAVE_EVERY_BATCHES, "batches")
print("Device:", device)

total_batches = len(train_loader)
batch_size = train_loader.batch_size

for epoch_index in range(start_epoch - 1, NUM_EPOCHS):

    current_epoch = epoch_index + 1

    # Restore current epoch progress if applicable
    if current_epoch == start_epoch:
        epoch_start_batch = start_batch
        epoch_running_loss = running_train_loss
        epoch_batches_done = batches_done
    else:
        epoch_start_batch = 0
        epoch_running_loss = 0.0
        epoch_batches_done = 0

    # -------------------------
    # TRAINING
    # -------------------------

    model.train()

    sampler = ResumableRandomSampler(
        train_loader.dataset,
        seed=SEED,
        epoch=epoch_index,
        start_index=epoch_start_batch * batch_size
    )

    epoch_train_loader = DataLoader(
        train_loader.dataset,
        batch_size=batch_size,
        sampler=sampler,
        num_workers=train_loader.num_workers,
        pin_memory=train_loader.pin_memory,
        collate_fn=train_loader.collate_fn
    )

    train_progress = tqdm(
        epoch_train_loader,
        total=total_batches - epoch_start_batch,
        desc=f"Epoch {current_epoch}/{NUM_EPOCHS} - Training"
    )

    for batch_offset, (images, masks) in enumerate(
        train_progress,
        start=epoch_start_batch
    ):

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            outputs = model(images)

        loss = criterion(outputs.float(), masks.float())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_running_loss += loss.item()
        epoch_batches_done += 1

        train_progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

        # -------------------------
        # SAVE EVERY 50 BATCHES
        # -------------------------

        if epoch_batches_done % SAVE_EVERY_BATCHES == 0:

            save_resume_checkpoint(
                epoch=current_epoch,
                next_batch=batch_offset + 1,
                running_train_loss=epoch_running_loss,
                batches_done=epoch_batches_done,
                best_val_loss=best_val_loss,
                history=history
            )

            train_progress.set_postfix(
                loss=f"{loss.item():.4f}",
                checkpoint="saved"
            )

    # -------------------------
    # VALIDATION
    # -------------------------

    epoch_train_loss = epoch_running_loss / total_batches

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():

        val_progress = tqdm(
            val_loader,
            desc=f"Epoch {current_epoch}/{NUM_EPOCHS} - Validation"
        )

        for images, masks in val_progress:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):
                outputs = model(images)

            loss = criterion(outputs.float(), masks.float())

            running_val_loss += loss.item()

            val_progress.set_postfix(
                loss=f"{loss.item():.4f}"
            )

    epoch_val_loss = running_val_loss / len(val_loader)

    # -------------------------
    # SAVE LATEST MODEL
    # -------------------------

    torch.save({
        "epoch": current_epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": epoch_train_loss,
        "val_loss": epoch_val_loss,
        "architecture": "ResNet34UNet",
        "image_size": 352
    }, LATEST_MODEL_PATH)

    # -------------------------
    # UPDATE BEST MODEL
    # -------------------------

    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss

        torch.save({
            "epoch": current_epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "architecture": "ResNet34UNet",
            "image_size": 352
        }, BEST_MODEL_PATH)

        checkpoint_status = "NEW BEST MODEL SAVED"

    else:
        checkpoint_status = "PREVIOUS BEST RETAINED"

    # -------------------------
    # RECORD HISTORY
    # -------------------------

    history.append({
        "epoch": current_epoch,
        "train_loss": epoch_train_loss,
        "val_loss": epoch_val_loss
    })

    with open(HISTORY_PATH, "w", newline="") as file:

        writer = csv.DictWriter(
            file,
            fieldnames=["epoch", "train_loss", "val_loss"]
        )

        writer.writeheader()
        writer.writerows(history)

    # -------------------------
    # SAVE EPOCH COMPLETION
    # -------------------------

    if current_epoch < NUM_EPOCHS:

        save_resume_checkpoint(
            epoch=current_epoch + 1,
            next_batch=0,
            running_train_loss=0.0,
            batches_done=0,
            best_val_loss=best_val_loss,
            history=history
        )
    else:

        save_resume_checkpoint(
            epoch=NUM_EPOCHS + 1,
            next_batch=0,
            running_train_loss=0.0,
            batches_done=0,
            best_val_loss=best_val_loss,
            history=history
        )

    # -------------------------
    # EPOCH SUMMARY
    # -------------------------

    print(f"\nEPOCH {current_epoch}/{NUM_EPOCHS}")
    print(f"Training loss: {epoch_train_loss:.6f}")
    print(f"Validation loss: {epoch_val_loss:.6f}")
    print(f"Best validation loss: {best_val_loss:.6f}")
    print("Latest model: SAVED")
    print("Best model:", checkpoint_status)
    print("Recovery checkpoint: SAVED")
    print("-" * 50)

# ==================================================
# 6. FINAL SUMMARY
# ==================================================

print("\nTRAINING FINISHED")

print("Best validation loss:", best_val_loss)
print("Best model exists:", BEST_MODEL_PATH.exists())
print("Latest model exists:", LATEST_MODEL_PATH.exists())
print("Recovery checkpoint exists:", RESUME_PATH.exists())
print("Training history exists:", HISTORY_PATH.exists())